# Project Context and Scope

This project is not intended to be a novel academic contribution.  
Instead, it reflects a **realistic, present-day business use case** where speed, interpretability, and deployability matter more than exhaustive experimentation.

As a result, several traditionally rigorous steps (such as extensive hyperparameter tuning, testing multiple model families, or formal statistical proofs) were intentionally skipped.  
These decisions were made to focus on:
- Practical demand forecasting
- Operational relevance
- Clear communication to non-technical stakeholders

The subsequent cells reflect these trade-offs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
bike_data = pd.read_csv('https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip', encoding='latin1')
bike_data.info()

In [ ]:
bike_data.columns = bike_data.columns.str.replace(' ', '_')
bike_data.columns = bike_data.columns.str.lower()

In [ ]:
#bike_data = bike_data.sort_values('date', ascending=False).reset_index()
#bike_data = bike_data.rename(columns={'index': 'rev'}).drop(columns='rev')

### Business Context and Constraints

Bike demand is highly variable and depends on:

- Time of day
- Seasonality
- Weather conditions
- Short-term momentum (recent demand patterns)

The business requirement is practical predictability, not academic novelty.
As a result, several traditionally rigorous steps (e.g., exhaustive hyperparameter tuning or multiple model comparisons) were intentionally skipped in favor of:

- Faster iteration
- Interpretability
- Realistic deployment assumptions

In [ ]:
bike_data['date'] = pd.to_datetime(bike_data['date'], errors='coerce')
bike_data['seasons'] = bike_data['seasons'].map({'Spring': 1, 'Summer': 2, 'Autumn': 3, 'Winter': 4})
bike_data['holiday'] = bike_data['holiday'].map({'Holiday': 1, 'No Holiday': 0})
bike_data['functioning_day'] = bike_data['functioning_day'].map({'Yes': 1, 'No': 0})
bike_data

In [ ]:
bike_data = bike_data[~bike_data['date'].isnull()]
bike_data

In [ ]:
throwaways = bike_data.drop(columns=['seasons', 'holiday', 'functioning_day'], inplace=True)

In [ ]:
future_pred_data = bike_data.copy()
future_pred_data # saved this for the future predictions, seeing as that is the business case (see last 3 cells)

### Data Cleaning and Initial Processing

The raw dataset contains a mix of numerical, categorical, and date-based features.

Initial preprocessing steps include:
- Standardizing column names for consistency
- Parsing date values into proper datetime objects
- Encoding categorical variables numerically

Some encoded categorical features are later removed as part of feature selection.  
This decision reflects a preference for **engineered time-based signals** over static categorical indicators, which often duplicate information already captured through cyclical encoding.

In [ ]:
days_in_year = bike_data['date'].dt.is_leap_year.map({True: 366, False: 365}) # neurotic check for leap years
date_theta = 2 * np.pi * (bike_data['date'].dt.day_of_year - 1) / days_in_year
hours_theta = 2 * np.pi * bike_data['hour'] / 24

### Time-Based Feature Engineering (Cyclical Encoding)

Time-related variables such as *hour of day* and *day of year* are inherently cyclical.  
For example:
- Hour 23 is closer to hour 0 than to hour 12
- December 31 is closer to January 1 than to July 1

To preserve these relationships, sine and cosine transformations are applied to:
- Day of year
- Hour of day

This allows the model to learn seasonal and daily patterns more naturally than with raw integer values.

In [ ]:
bike_data['date_sin'] = np.sin(date_theta)
bike_data['date_cos'] = np.cos(date_theta)
bike_data['hour_sin'] = np.sin(hours_theta)
bike_data['hour_cos'] = np.cos(hours_theta)
bike_data['temp_hour'] = bike_data['temperature(°c)'] / 24
bike_data

### Lagged Demand and Trend Features

Bike demand is strongly influenced by **recent usage behavior**.

To capture short-term momentum, several lag-based features are introduced:
- Previous demand values (1, 2, and 3 periods prior)
- Rolling mean to smooth sharp fluctuations
- Rolling standard deviation to capture volatility
- A simple trend indicator measuring recent change

The rolling window sizes are intentionally short.  
This reflects an assumption that **recent demand patterns matter more operationally** than long-term historical averages.
Especially when accounting for things like holidays and weekends, as well as compensating for when demand is quite low;
often after midnight or in more traceable cases towards the end of 2017.

These choices are based on practical intuition as someone who used ride sharing services myself,
and observed ride-hailing behavior, rather than formal statistical validation.

NOTE: I am aware of my bias and how it MAY introduce overfitting

In [ ]:
bike_data['rbc_1'] = bike_data['rented_bike_count'].shift(1) # shifts by 1 day
bike_data['rbc_2'] = bike_data['rented_bike_count'].shift(2) # shifts by 2 days
bike_data['rbc_3'] = bike_data['rented_bike_count'].shift(3) # shifts by 3 days
bike_data['rbc_rolling_mean'] = bike_data['rented_bike_count'].shift(1).rolling(window=7).mean() # this is to smoothen the data
bike_data['rbc_rolling_std'] = bike_data['rented_bike_count'].shift(1).rolling(window=7).std() # this is to account for variance in trend, if at all
bike_data['rbc_trend'] = bike_data['rented_bike_count'].rolling(3).apply(lambda x: x.iloc[-1] - x.iloc[0])

In [ ]:
bike_data = bike_data.dropna().reset_index(drop=True)
bike_data_num = bike_data.select_dtypes(include=np.number)

In [ ]:
bike_data_num.columns

In [ ]:
feature_col = [
    'hour', 'temperature(°c)','humidity(%)',
    'wind_speed_(m/s)', 'visibility_(10m)', 'dew_point_temperature(°c)',
    'solar_radiation_(mj/m2)', 'rainfall(mm)', 'snowfall_(cm)', 'date_sin', 'date_cos', 'hour_sin',
    'hour_cos', 'temp_hour', 'rbc_1', 'rbc_2', 'rbc_3', 'rbc_rolling_mean',
    'rbc_rolling_std', 'rbc_trend']

target = 'rented_bike_count'

bike_feature = bike_data_num[feature_col]
bike_label = bike_data_num[target]

split = int(len(bike_data_num) * 0.7)

### Train–Test Split Strategy

Data split chronologically (70% train, 30% test)

No shuffling, also this is complementary to Keras' validation_split

*Why this matters:*
This mirrors real-world deployment, where future data is never known at training time.

In [ ]:
feature_train = bike_feature.iloc[:split]
feature_test = bike_feature.iloc[split:]

label_train = bike_label.iloc[:split]
label_test = bike_label.iloc[split:]

In [ ]:
label_test.info()

In [ ]:
import tensorflow as tf

SEED_VALUE = 42
 
# Fix seed to make training deterministic.
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

### Model Architecture and Design Choices

A feedforward neural network is used for prediction.

Key design considerations:
- Non-linear relationships between weather, time, and demand
- Relatively low architectural complexity for easier deployment
- Use of Mean Absolute Error (MAE), which aligns closely with business impact

The model architecture favors stability and interpretability over maximum theoretical performance.

In [ ]:
import keras

def build_model(n_features):
    model = keras.Sequential()
    model.add(keras.layers.Dense(32, activation='gelu', input_shape = (n_features,)))
    #model.add(keras.layers.Dense(64, activation='gelu'))
    model.add(keras.layers.Dense(16, activation='gelu'))
    model.add(keras.layers.Dense(1))
    
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='mean_absolute_error', metrics=['mse', 'mae'])
    
    return model

### Model Training Behavior

Training and validation loss curves are monitored to assess convergence.

Early stopping is applied to:
- Prevent overfitting
- Reduce unnecessary computation
- Maintain generalization performance

The observed loss trends indicate stable learning without significant divergence between training and validation sets.


In [ ]:
bike_model = build_model(len(feature_col))
history = bike_model.fit(feature_train, label_train, epochs=500, batch_size=1024, validation_split=0.2, verbose=0, shuffle=False, callbacks=keras.callbacks.EarlyStopping(patience=40))

In [ ]:
loss, mae1, val_loss, val_mae = history.history['loss'][-1], history.history['mae'][-1], history.history['val_loss'][-1], history.history['val_mae'][-1]
loss, mae1, val_loss, val_mae

### Why use an R² score?

Firstly because stakeholders find it easier to digest "accuracy" of the model when expressed in percentages (r2_score * 100),
hence its painful (to other practitioners) usage here. It is not the best metric to judge, some may say, but a non-ML person wouldn't know any better
as far as the nerd olympics go on this subject matter, so this is the most tangible abstraction for them.

The high R² score is driven largely by strong temporal structure and short-term demand persistence in the data.
This indicates that bike usage patterns in Seoul are highly regular and predictable at an hourly level when recent demand information is available.

It is also worthy of note that this is only possible due to the lag and rolling features provided. They provide correlation for spotting trends.
As it is with time-series data, if this were to be frequentlyt divergent in trend, it will reflect in the image below - it would be horrific, like a seismic signal.

In [ ]:
from sklearn.metrics import r2_score

label_pred = bike_model.predict(feature_test).flatten()
r2 = r2_score(label_test, label_pred)

print('r2 score: ', r2)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlim([0, 400])
plt.ylim([0, 200])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

In [ ]:
results = pd.DataFrame({
    'actual': label_test.values,
    'predicted': label_pred.round()})

results['diff'] = (results['predicted'] - results['actual']).round()

results

# FUTURE PREDICTIONS

From here on out we define new features using existing ones, and use them to predict future hourly demand.
The safest way to do this (I think) is since the `rented_bike_count` for the new is obviously unknown, there will be a shortage in the rolling rbc's.
Consequently, given that all other weather features are unknown as well, it seemed right to use means of each hour's value in correspondence.

I would assume (maybe even argue) that this creates some sort of artiicial trend or relationship between the hours in each day and what possible temperatures they might be,
unbound by season of course, therefore treating each as a separate case. Hour by hour.

This also serves as a decent level of chaos to test the model with, since the `rbc_1` exists in a state of *some* kind of unique randomness from hour 0 to hour 23.

Finally, the end result being a dataframe that consists of upper and lower limits allow for flexibility within the Mean Absolute Error for each hourly prediction.

In [ ]:
import datetime as dt
from datetime import timedelta

In [ ]:
bike_data_copy = future_pred_data.copy()
bike_data_copy.tail(5)

In [ ]:
bike_data_copy['rbc_1'] = bike_data_copy['rented_bike_count'].shift(1) 

In [ ]:
hour_list = []
date_list = []
temp = []
hum = []
wind = []
vis = []
dew = []
solar = []
rain = []
snow = []
rbc_1 = []


for i in range(24):
    hour_list.append(i)
    date_list.append(bike_data_copy['date'].iloc[-1] + timedelta(1))
    
    agg_data = bike_data_copy[bike_data_copy['hour'] == i]
    
    agg_temperature = agg_data['temperature(°c)'].mean()
    temp.append(agg_temperature)
    
    agg_humidity = agg_data['humidity(%)'].mean()
    hum.append(agg_humidity)
    
    agg_wind_speed = agg_data['wind_speed_(m/s)'].mean()
    wind.append(agg_wind_speed)
    
    agg_visibility = agg_data['visibility_(10m)'].mean()
    vis.append(agg_visibility)
    
    agg_dew_point_temperature = agg_data['dew_point_temperature(°c)'].mean()
    dew.append(agg_dew_point_temperature)
    
    agg_solar_radiation = agg_data['solar_radiation_(mj/m2)'].mean()
    solar.append(agg_solar_radiation)
    
    agg_rainfall = agg_data['rainfall(mm)'].mean()
    rain.append(agg_rainfall)
    
    agg_snowfall = agg_data['snowfall_(cm)'].mean()
    snow.append(agg_snowfall)
    
    agg_rbc_1 = agg_data['rbc_1'].mean()
    rbc_1.append(agg_rbc_1)
    
feats = pd.DataFrame({'hour': hour_list, 'date': date_list,
                      'temperature(°c)': temp, 'humidity(%)': hum,
                      'wind_speed_(m/s)': wind, 'visibility_(10m)': vis,
                      'dew_point_temperature(°c)': dew, 'solar_radiation_(mj/m2)': solar,
                      'rainfall(mm)': rain, 'snowfall_(cm)': snow,
                      'rbc_1': rbc_1}) 

bike_data_copy = pd.concat([bike_data_copy, feats], ignore_index=True)

In [ ]:
bike_data_copy['rbc_2'] = bike_data_copy['rbc_1'].shift(1) 
bike_data_copy['rbc_3'] = bike_data_copy['rbc_1'].shift(2) 
bike_data_copy['rbc_rolling_mean'] = bike_data_copy['rbc_1'].shift(1).rolling(window=7).mean() 
bike_data_copy['rbc_rolling_std'] = bike_data_copy['rbc_1'].shift(1).rolling(window=7).std() 
bike_data_copy['rbc_trend'] = bike_data_copy['rbc_1'].rolling(3).apply(lambda x: x.iloc[-1] - x.iloc[0])

#bike_data_copy = bike_data_copy.dropna().reset_index(drop=True)
#bike_data_copy = bike_data_copy.select_dtypes(include=np.number)

In [ ]:
bike_data_copy.fillna(np.nan, inplace=True)

In [ ]:
bike_data_copy.tail(24)

In [ ]:
day_in_year = bike_data_copy['date'].dt.is_leap_year.map({True: 366, False: 365}) # neurotic check for leap years
dt_theta = 2 * np.pi * (bike_data_copy['date'].dt.day_of_year - 1) / day_in_year
hr_theta = 2 * np.pi * bike_data_copy['hour'] / 24

bike_data_copy['date_sin'] = np.sin(dt_theta)
bike_data_copy['date_cos'] = np.cos(dt_theta)
bike_data_copy['hour_sin'] = np.sin(hr_theta)
bike_data_copy['hour_cos'] = np.cos(hr_theta)
bike_data_copy['temp_hour'] = bike_data_copy['temperature(°c)'] / 24

In [ ]:
bike_data_copy.tail(24)

In [ ]:
pred_data = bike_data_copy[feature_col][-24:]
pred_data

In [ ]:
import tensorflow as tf

SEED_VAL = 24
 
np.random.seed(SEED_VAL)
tf.random.set_seed(SEED_VAL)

In [ ]:
feature_test

In [ ]:
next_pred = bike_model.predict(pred_data).flatten().round(0)
hours = np.array(bike_data_copy['hour'].iloc[-24:]).flatten()
pred_results = pd.DataFrame({'predictions': pd.Series(next_pred), 'hour': hours})

In [ ]:
pred_results

In [ ]:
pred_results['lower_limit'] = (pred_results['predictions'] - mae).round(0)
pred_results['upper_limit'] = (pred_results['predictions'] + mae).round(0)

In [ ]:
print(f"{'='*5} Next Day hourly bike demand future predictions {'='*5}")
pred_results

In [616]:
bike_data_num.describe()

,rented_bike_count,hour,temperature(°c),humidity(%),wind_speed_(m/s),visibility_(10m),dew_point_temperature(°c),solar_radiation_(mj/m2),rainfall(mm),snowfall_(cm),...,date_cos,hour_sin,hour_cos,temp_hour,rbc_1,rbc_2,rbc_3,rbc_rolling_mean,rbc_rolling_std,rbc_trend
count,3449.000000,3449.000000,3449.000000,3449.000000,3449.00000,3449.000000,3449.000000,3449.000000,3449.000000,3449.000000,...,3449.000000,3449.000000,3.449000e+03,3449.000000,3449.000000,3449.000000,3449.000000,3449.000000,3449.000000,3449.000000
mean,674.806321,11.517251,12.429226,58.974775,1.81154,1514.247608,3.883271,0.572343,0.158191,0.029255,...,0.001325,-0.001246,-1.246123e-03,0.517884,674.655262,674.387649,674.065526,673.637742,275.344119,0.418672
std,653.669652,6.919013,12.221654,19.997004,1.11834,580.191808,13.329567,0.867518,1.035079,0.225324,...,0.705425,0.707208,7.072082e-01,0.509236,653.723563,653.769843,653.789915,566.642937,219.055094,436.334211
min,0.000000,0.000000,-15.100000,14.000000,0.00000,66.000000,-26.900000,0.000000,0.000000,0.000000,...,-0.999963,-1.000000,-1.000000e+00,-0.629167,0.000000,0.000000,0.000000,0.000000,0.000000,-1760.000000
25%,171.000000,6.000000,2.400000,44.000000,1.00000,1153.000000,-6.400000,0.000000,0.000000,0.000000,...,-0.766659,-0.707107,-7.071068e-01,0.100000,171.000000,170.000000,170.000000,216.000000,90.529395,-172.000000
50%,440.000000,12.000000,13.700000,57.000000,1.60000,1805.000000,6.400000,0.010000,0.000000,0.000000,...,0.004304,0.000000,-1.836970e-16,0.570833,438.000000,438.000000,438.000000,500.571429,221.241196,0.000000
75%,1041.000000,18.000000,21.900000,75.000000,2.40000,2000.000000,14.100000,0.950000,0.000000,0.000000,...,0.651899,0.707107,7.071068e-01,0.912500,1041.000000,1041.000000,1041.000000,1046.000000,422.335175,147.000000
max,3404.000000,23.000000,39.400000,98.000000,7.40000,2000.000000,27.200000,3.420000,24.000000,4.300000,...,1.000000,1.000000,1.000000e+00,1.641667,3404.000000,3404.000000,3404.000000,2522.000000,996.879082,1831.000000
